# Test 1: Marchenko-Pastur Spectral Structure Hypothesis Validation

**Framleis Law Falsification Roadmap — Test 1 of 6**

**Prediction:** Pre-trained system weights show lower spectral entropy (higher τ structure) than random Gaussian matrices of equal size.

**Null Hypothesis:** W_pretrained has same eigenvalue distribution as random Gaussian W_random ~ N(0, 1/n)

**Test Method:** Marchenko-Pastur KS-test with bootstrap p-value calibration (n_bootstrap=500)

**Failure Criterion:** Random and structured weights have indistinguishable spectral entropy distributions.

---

## Setup: Load mp_test.py from Tofoo repo


In [ ]:
# Clone repo (private)
import subprocess
import os

repo_path = '/tmp/tofoo_mp_test'
if not os.path.exists(repo_path):
    # For private repo in Colab, user must authenticate
    # In local/RunPod environment, use local path
    print("Checking for local mp_test.py...")
    if os.path.exists('/home/user/Tofoo-/mp_test.py'):
        # Local environment
        import sys
        sys.path.insert(0, '/home/user/Tofoo-')
        print("Using local path: /home/user/Tofoo-")
    else:
        print("ERROR: mp_test.py not found. For Colab, see CLAUDE.md: open File→Open notebook→GitHub tab, search nsolland/Tofoo-")
        print("Then manually copy mp_test.py into notebook directory.")
else:
    import sys
    sys.path.insert(0, repo_path)


In [ ]:
# Import Marchenko-Pastur test functions
from mp_test import mp_test_ks, effective_rank_test, marchenko_pastur_pdf, marchenko_pastur_cdf

import numpy as np
import torch
from transformers import AutoModel
import matplotlib.pyplot as plt
import math

# Goldilocks zone from Framleis Law
GAMMA_EC = 0.5772156649
ZETA3 = 1.2020569032
TAU_MIN = math.exp(-GAMMA_EC)  # 0.5615
TAU_MAX = 1.0 / ZETA3          # 0.8319

print(f"Goldilocks zone: [{TAU_MIN:.4f}, {TAU_MAX:.4f}]")
print("Test 1 imports ready.")


## Test 1a: Random Gaussian Baseline (Null Distribution)


In [ ]:
# Generate random Gaussian matrix (null hypothesis)
# Size: same as GPT-2 embedding projection (~768 → 768)

np.random.seed(42)
m, n = 768, 768

# Create random Gaussian weight matrix W ~ N(0, 1/n)
W_random = np.random.randn(m, n) / np.sqrt(n)

print(f"Random matrix shape: {W_random.shape}")
print(f"Random matrix mean: {W_random.mean():.6f}")
print(f"Random matrix std:  {W_random.std():.6f}")
print()

# Test 1a: MP KS-test on random matrix
print("[TEST 1a: RANDOM BASELINE]")
rejected_mp_r, p_mp_r, info_mp_r = mp_test_ks(W_random, n_bootstrap=500, verbose=True)
print()

# Test 1a: Effective rank test on random matrix
print("[TEST 1a: EFFECTIVE RANK TEST - RANDOM]")
rejected_er_r, p_er_r, info_er_r = effective_rank_test(W_random, n_bootstrap=500, verbose=True)


## Test 1b: Pre-trained GPT-2 Weights


In [ ]:
# Load GPT-2 and extract weight matrices
print("Loading GPT-2 (117M)...")
model = AutoModel.from_pretrained('gpt2', output_hidden_states=True)
model.eval()
print("GPT-2 loaded.\n")

# Collect all 2D weight matrices
weights = {}
for name, param in model.named_parameters():
    if param.dim() == 2 and min(param.shape) >= 64:
        W_np = param.detach().float().numpy()
        weights[name] = W_np

print(f"Found {len(weights)} weight matrices >= 64×64")
print("\nFirst 5 matrices:")
for i, name in enumerate(list(weights.keys())[:5]):
    print(f"  {name:<50} shape={weights[name].shape}")


In [ ]:
# Test on first layer (typically attention projection)
first_layer_name = list(weights.keys())[0]
W_pretrained = weights[first_layer_name]

print(f"Testing layer: {first_layer_name}")
print(f"Shape: {W_pretrained.shape}")
print(f"Mean: {W_pretrained.mean():.6f}")
print(f"Std:  {W_pretrained.std():.6f}")
print()

# Normalize to match null hypothesis (W ~ N(0, 1/n))
n = W_pretrained.shape[1]
W_norm = W_pretrained / np.sqrt(n)

print("[TEST 1b: PRE-TRAINED WEIGHTS (NORMALIZED)]")
rejected_mp_p, p_mp_p, info_mp_p = mp_test_ks(W_norm, n_bootstrap=500, verbose=True)
print()

print("[TEST 1b: EFFECTIVE RANK TEST - PRE-TRAINED]")
rejected_er_p, p_er_p, info_er_p = effective_rank_test(W_norm, n_bootstrap=500, verbose=True)


## Test 1c: Results Summary


In [ ]:
import pandas as pd

# Summary table
results_df = pd.DataFrame({
    'Test': ['Random Baseline', 'Pre-trained GPT-2'],
    'MP KS p-value': [p_mp_r, p_mp_p],
    'MP Rejected?': [rejected_mp_r, rejected_mp_p],
    'ER p-value': [p_er_r, p_er_p],
    'ER Rejected?': [rejected_er_r, rejected_er_p],
})

print("\n" + "="*80)
print("TEST 1: MARCHENKO-PASTUR SPECTRAL STRUCTURE HYPOTHESIS")
print("="*80)
print(results_df.to_string(index=False))
print()

# Interpretation
print("INTERPRETATION:")
print()
print(f"Random Baseline:")
print(f"  MP KS: p={p_mp_r:.4f} → Null {'REJECTED (error)' if rejected_mp_r else 'NOT REJECTED (correct)'}")
print(f"  ER:    p={p_er_r:.4f} → Null {'REJECTED (error)' if rejected_er_r else 'NOT REJECTED (correct)'}")
print()
print(f"Pre-trained GPT-2:")
print(f"  MP KS: p={p_mp_p:.4f} → Null {'REJECTED (correct)' if rejected_mp_p else 'NOT REJECTED (error)'}")
print(f"  ER:    p={p_er_p:.4f} → Null {'REJECTED (correct)' if rejected_er_p else 'NOT REJECTED (error)'}")
print()

# Test 1 verdict
test1_pass = (not rejected_mp_r) and rejected_mp_p and (not rejected_er_r) and rejected_er_p
verdict = "✓ PASS" if test1_pass else "✗ FAIL"
print(f"Test 1 Verdict: {verdict}")
print()
if test1_pass:
    print("CONCLUSION: Random and pre-trained weights have DIFFERENT spectral distributions.")
    print("Framleis prediction SUPPORTED: Pre-trained weights are structured.")
else:
    print("CONCLUSION: Cannot distinguish random from pre-trained.")
    print("Framleis prediction CONTRADICTED.")


## Test 1d: Extended Validation (All GPT-2 Layers)


In [ ]:
# Run MP test on multiple layers
test_results = []

print(f"Testing {len(weights)} weight matrices...\n")

for layer_name, W in list(weights.items())[:10]:  # First 10 layers to save time
    # Normalize
    n = W.shape[1]
    W_norm = W / np.sqrt(n)
    
    # Test (fewer bootstraps for speed)
    rejected_mp, p_mp, _ = mp_test_ks(W_norm, n_bootstrap=200, verbose=False)
    rejected_er, p_er, _ = effective_rank_test(W_norm, n_bootstrap=200, verbose=False)
    
    test_results.append({
        'Layer': layer_name[:50],
        'Shape': f"{W.shape[0]}×{W.shape[1]}",
        'MP p': p_mp,
        'MP Rej': rejected_mp,
        'ER p': p_er,
        'ER Rej': rejected_er,
    })
    
    print(f"{'✓' if rejected_mp else '✗'} MP={p_mp:.4f}  "
          f"{'✓' if rejected_er else '✗'} ER={p_er:.4f}  "
          f"{layer_name[:50]}")

# Summary
df_extended = pd.DataFrame(test_results)
n_mp_reject = df_extended['MP Rej'].sum()
n_er_reject = df_extended['ER Rej'].sum()

print()
print(f"Summary: {n_mp_reject}/{len(df_extended)} layers rejected by MP test")
print(f"Summary: {n_er_reject}/{len(df_extended)} layers rejected by ER test")
print()
if n_mp_reject >= len(df_extended) * 0.8:
    print("✓ STRONG: Most GPT-2 layers are significantly different from random.")
elif n_mp_reject >= len(df_extended) * 0.5:
    print("~ MODERATE: About half of GPT-2 layers are structured.")
else:
    print("✗ WEAK: Few GPT-2 layers show structure.")


## Test 1e: Visualization


In [ ]:
# Visualize eigenvalue distributions
import scipy.stats

# Compute eigenvalues for both cases
def get_eigenvalues(W):
    """Get eigenvalues of WW^T (sorted descending)"""
    gram = W @ W.T
    evals = np.linalg.eigvalsh(gram)
    evals = evals[evals > 1e-10]
    return np.sort(evals)[::-1]

evals_random = get_eigenvalues(W_random / np.sqrt(W_random.shape[1]))
evals_pretrained = get_eigenvalues(W_norm)

# Plot
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Random
ax = axes[0]
ax.hist(evals_random, bins=50, density=True, alpha=0.7, label='Empirical', color='blue')
ax.set_xlabel('Eigenvalue λ')
ax.set_ylabel('Density')
ax.set_title(f'Random Gaussian (N={len(evals_random)})')
ax.legend()
ax.grid(alpha=0.3)

# Pre-trained
ax = axes[1]
ax.hist(evals_pretrained, bins=50, density=True, alpha=0.7, label='Empirical', color='green')
ax.set_xlabel('Eigenvalue λ')
ax.set_ylabel('Density')
ax.set_title(f'Pre-trained GPT-2 (N={len(evals_pretrained)})')
ax.legend()
ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('test1_eigenvalue_distributions.png', dpi=100, bbox_inches='tight')
plt.show()

print("Saved: test1_eigenvalue_distributions.png")


## Test 1 Summary & Next Steps


In [ ]:
print("="*80)
print("FALSIFICATION TEST 1: COMPLETE")
print("="*80)
print()
print("Hypothesis:")
print("  Pre-trained system weights show lower spectral entropy than random Gaussian")
print()
print("Results:")
print(f"  Random baseline (null): MP p={p_mp_r:.4f}, not rejected ✓")
print(f"  Pre-trained GPT-2: MP p={p_mp_p:.4f}, rejected ✓")
print()
if not rejected_mp_r and rejected_mp_p:
    print("✓ TEST 1 PASSED: Prediction supported")
    print()
    print("Next steps (Test 2-6):")
    print("  - Test 2: Model scaling law (Qwen2.5-14B, 32B, 70B)")
    print("  - Test 3: Frozen-core mapping (layer-by-layer τ)")
    print("  - Test 4: Margin responsivity (learning curves)")
    print("  - Test 5: Degradation under error")
    print("  - Test 6: Bifurcation analysis (completed in theory)")
else:
    print("✗ TEST 1 FAILED")
print()
print("="*80)
